In [ ]:
!pip -q install transformers torch scikit-learn matplotlib

In [ ]:
!pip -q install gradio pillow

In [ ]:
!pip -q install fastapi uvicorn pyngrok nest_asyncio

In [2]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!mv cloudflared /usr/local/bin

--2026-05-28 08:06:53--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
Resolving github.com (github.com)... 140.82.121.4
Connecting to github.com (github.com)|140.82.121.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.5.2/cloudflared-linux-amd64 [following]
--2026-05-28 08:06:53--  https://github.com/cloudflare/cloudflared/releases/download/2026.5.2/cloudflared-linux-amd64
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/49ed726b-742b-4dbe-a0cf-d9efc4d22773?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-05-28T09%3A02%3A43Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-05-28T0

In [5]:
!which cloudflared

/usr/local/bin/cloudflared


In [8]:
# ============================================================
# LSML LATENT PROJECTOR RUNTIME

import json
import math
import random
import re
import socket
import subprocess
import threading
import time

import nest_asyncio
import numpy as np
import torch
import torch.nn.functional as F

from fastapi import FastAPI, WebSocket
from fastapi.responses import HTMLResponse

from sklearn.decomposition import PCA

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer
)

import uvicorn


# ============================================================
# PATCH LOOP
# ============================================================

nest_asyncio.apply()


# ============================================================
# DEVICE
# ============================================================

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("DEVICE:", DEVICE)


# ============================================================
# MODEL
# ============================================================

MODEL_NAME = "gpt2"

TOKENIZER = AutoTokenizer.from_pretrained(MODEL_NAME)

MODEL = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    output_hidden_states=True,
    output_attentions=True
).to(DEVICE)

MODEL.eval()


# ============================================================
# CONFIG
# ============================================================

WIDTH = 1600
HEIGHT = 900
MAX_BODIES = 48


# ============================================================
# GLOBAL STATE
# ============================================================

STATE = {

    "nodes": [],

    "text": "",

    "latent": None,

    "bias_vector": None

}


# ============================================================
# FREE PORT
# ============================================================


def find_free_port():

    s = socket.socket()

    s.bind(("", 0))

    port = s.getsockname()[1]

    s.close()

    return port


PORT = find_free_port()

print("PORT:", PORT)


# ============================================================
# TOKEN COLOR
# ============================================================


def token_color(token_id):

    rng = random.Random(token_id)

    r = rng.randint(100,255)
    g = rng.randint(100,255)
    b = rng.randint(100,255)

    return f"rgb({r},{g},{b})"


# ============================================================
# LATENT PROJECTOR
# ============================================================


def latent_projector(
    hidden_states,
    attentions,
    token_ids
):

    latents = (
        hidden_states[-1][0]
        .detach()
        .cpu()
        .numpy()
    )

    count = min(MAX_BODIES, latents.shape[0])

    latents = latents[-count:]

    pca = PCA(n_components=2)

    points = pca.fit_transform(latents)

    px = points[:,0]
    py = points[:,1]

    px = (px - px.min()) / (px.max() - px.min() + 1e-6)
    py = (py - py.min()) / (py.max() - py.min() + 1e-6)

    px = px * (WIDTH - 200) + 100
    py = py * (HEIGHT - 200) + 100

    attn = attentions[-1][0].mean(dim=0)

    nodes = []

    for i in range(count):

        token_id = int(token_ids[-count+i])

        token = TOKENIZER.decode([token_id])

        mass = float(attn[i].mean().item())

        size = 18 + mass * 240

        angle = random.uniform(0, math.pi * 2)

        radius = size

        verts = []

        for j in range(3):

            a = angle + j * math.pi * 2 / 3

            vx = px[i] + math.cos(a) * radius
            vy = py[i] + math.sin(a) * radius

            verts.append([
                float(vx),
                float(vy)
            ])

        nodes.append({

            "id": i,

            "token": token,

            "token_id": token_id,

            "x": float(px[i]),

            "y": float(py[i]),

            "size": float(size),

            "mass": float(mass),

            "angle": float(angle),

            "vertices": verts,

            "color": token_color(token_id),

            "embedding": latents[i].tolist()

        })

    return nodes


# ============================================================
# SEMANTIC FIELD DELTA
# ============================================================


def semantic_field_delta(nodes):

    vectors = []

    for node in nodes:

        emb = np.array(node["embedding"])

        nx = node["x"] / WIDTH
        ny = node["y"] / HEIGHT

        position_force = (
            (nx - 0.5) * 0.6 +
            (ny - 0.5) * 0.4
        )

        scale_force = node["size"] / 120.0

        mass_force = node["mass"] * 2.0

        total_force = (
            1.0 +
            position_force +
            scale_force +
            mass_force
        )

        emb = emb * total_force

        vectors.append(emb)

    if len(vectors) == 0:
        return None

    latent_bias = np.mean(vectors, axis=0)

    latent_bias = latent_bias / (
        np.linalg.norm(latent_bias) + 1e-6
    )

    return latent_bias


# ============================================================
# GENERATION
# ============================================================


def generate(prompt):

    inputs = TOKENIZER(
        prompt,
        return_tensors="pt"
    ).to(DEVICE)

    generated = inputs["input_ids"]

    latent_bias = STATE.get("bias_vector")

    if latent_bias is not None:

        latent_bias = torch.tensor(
            latent_bias,
            dtype=torch.float32,
            device=DEVICE
        )

    for step in range(60):

        with torch.no_grad():

            outputs = MODEL(
                generated,
                output_hidden_states=True,
                output_attentions=True
            )

        logits = outputs.logits[:, -1, :]

        if latent_bias is not None:

            hidden = outputs.hidden_states[-1][:, -1, :]

            similarity = F.cosine_similarity(
                hidden,
                latent_bias.unsqueeze(0),
                dim=-1
            )

            similarity = similarity.mean()

            logits = logits * (
                1.0 + similarity * 0.35
            )

        probs = F.softmax(logits, dim=-1)

        next_token = torch.multinomial(
            probs,
            num_samples=1
        )

        generated = torch.cat(
            [generated, next_token],
            dim=1
        )

    with torch.no_grad():

        outputs = MODEL(
            generated,
            output_hidden_states=True,
            output_attentions=True
        )

    token_ids = (
        generated[0]
        .cpu()
        .numpy()
        .tolist()
    )

    text = TOKENIZER.decode(
        token_ids,
        skip_special_tokens=True
    )

    nodes = latent_projector(
        outputs.hidden_states,
        outputs.attentions,
        token_ids
    )

    STATE["nodes"] = nodes
    STATE["text"] = text


# ============================================================
# FASTAPI
# ============================================================

app = FastAPI()


# ============================================================
# HTML
# ============================================================

HTML = """
<!DOCTYPE html>
<html>
<head>
<meta name='viewport' content='width=device-width, initial-scale=1.0'>
<title>LSML Runtime</title>
</head>
<body style='margin:0;overflow:hidden;background:#111;'>

<div style='
position:absolute;
top:0;
left:0;
right:0;
height:70px;
background:#1b1b1b;
display:flex;
gap:10px;
padding:10px;
z-index:10;
'>

<input
id='prompt'
value='Explain semantic topology and latent reasoning'
style='
flex:1;
background:#222;
color:white;
border:none;
padding:12px;
font-size:18px;
border-radius:14px;
'
/>

<button
id='generate'
style='
padding:12px 18px;
border:none;
background:#4c7dff;
color:white;
border-radius:14px;
font-size:16px;
cursor:pointer;
'
>
Generate
</button>

</div>

<canvas
id='canvas'
style='
position:absolute;
top:70px;
left:0;
width:100vw;
height:calc(100vh - 70px);
'
></canvas>

<script>

const canvas = document.getElementById('canvas');
const ctx = canvas.getContext('2d');

let nodes = [];
let selected = null;

function resize(){

    canvas.width = window.innerWidth;
    canvas.height = window.innerHeight - 70;
}

window.onresize = resize;
resize();

const protocol = (
    location.protocol === 'https:'
    ? 'wss'
    : 'ws'
);

const ws = new WebSocket(
    `${protocol}://${location.host}/ws`
);

ws.onmessage = event => {

    const data = JSON.parse(event.data);

    nodes = data.nodes;

    draw();
};


document.getElementById('generate').onclick = () => {

    const prompt = document.getElementById('prompt').value;

    ws.send(JSON.stringify({

        type:'generate',

        prompt:prompt

    }));
};


function drawConnections(){

    for(let i=0;i<nodes.length;i++){

        for(let j=i+1;j<nodes.length;j++){

            const a = nodes[i];
            const b = nodes[j];

            const dx = b.x - a.x;
            const dy = b.y - a.y;

            const dist = Math.sqrt(dx*dx + dy*dy);

            if(dist < 220){

                ctx.strokeStyle = 'rgba(255,255,255,0.07)';

                ctx.beginPath();

                ctx.moveTo(a.x,a.y);

                ctx.lineTo(b.x,b.y);

                ctx.stroke();
            }
        }
    }
}


function drawBody(node){

    ctx.save();

    const grad = ctx.createRadialGradient(
        node.x,
        node.y,
        0,
        node.x,
        node.y,
        node.size * 3
    );

    grad.addColorStop(0, node.color + 'AA');
    grad.addColorStop(1, node.color + '00');

    ctx.fillStyle = grad;

    ctx.beginPath();

    ctx.arc(
        node.x,
        node.y,
        node.size * 3,
        0,
        Math.PI * 2
    );

    ctx.fill();

    const verts = node.vertices;

    ctx.beginPath();

    ctx.moveTo(verts[0][0], verts[0][1]);

    ctx.lineTo(verts[1][0], verts[1][1]);

    ctx.lineTo(verts[2][0], verts[2][1]);

    ctx.closePath();

    ctx.fillStyle = node.color;

    ctx.shadowBlur = 30;
    ctx.shadowColor = node.color;

    ctx.fill();

    ctx.fillStyle = 'white';

    ctx.font = '16px sans-serif';

    ctx.fillText(
        node.token,
        node.x + 10,
        node.y - 10
    );

    ctx.restore();
}


function draw(){

    ctx.clearRect(
        0,
        0,
        canvas.width,
        canvas.height
    );

    drawConnections();

    for(const node of nodes){

        drawBody(node);
    }
}


function dist(ax,ay,bx,by){

    return Math.sqrt(
        (ax-bx)*(ax-bx)+
        (ay-by)*(ay-by)
    );
}


canvas.onpointerdown = e => {

    const rect = canvas.getBoundingClientRect();

    const mx = e.clientX - rect.left;
    const my = e.clientY - rect.top;

    for(const node of nodes){

        if(
            dist(mx,my,node.x,node.y)
            < node.size
        ){
            selected = node;
            break;
        }
    }
};


canvas.onpointermove = e => {

    if(!selected)
        return;

    const rect = canvas.getBoundingClientRect();

    selected.x = e.clientX - rect.left;
    selected.y = e.clientY - rect.top;

    const radius = selected.size;

    for(let i=0;i<3;i++){

        const a = (
            selected.angle +
            i * Math.PI * 2 / 3
        );

        selected.vertices[i][0] = (
            selected.x +
            Math.cos(a) * radius
        );

        selected.vertices[i][1] = (
            selected.y +
            Math.sin(a) * radius
        );
    }

    draw();

    ws.send(JSON.stringify({

        type:'update_nodes',

        nodes:nodes

    }));
};


canvas.onpointerup = () => {

    selected = null;
};


canvas.onwheel = e => {

    if(!selected)
        return;

    e.preventDefault();

    selected.size += e.deltaY * -0.05;

    selected.size = Math.max(
        10,
        Math.min(250, selected.size)
    );

    draw();

    ws.send(JSON.stringify({

        type:'update_nodes',

        nodes:nodes

    }));
};

</script>
</body>
</html>
"""


# ============================================================
# ROUTES
# ============================================================

@app.get("/")
async def root():

    return HTMLResponse(HTML)


# ============================================================
# WEBSOCKET
# ============================================================

@app.websocket("/ws")
async def websocket(ws: WebSocket):

    await ws.accept()

    while True:

        data = await ws.receive_text()

        payload = json.loads(data)

        if payload["type"] == "generate":

            generate(payload["prompt"])

            await ws.send_text(
                json.dumps({

                    "nodes": STATE["nodes"],

                    "text": STATE["text"]

                })
            )

        elif payload["type"] == "update_nodes":

            STATE["nodes"] = payload["nodes"]

            STATE["bias_vector"] = semantic_field_delta(
                STATE["nodes"]
            )


# ============================================================
# SERVER
# ============================================================


def run_server():

    uvicorn.run(
        app,
        host="0.0.0.0",
        port=PORT,
        log_level="warning"
    )


threading.Thread(
    target=run_server,
    daemon=True
).start()


# ============================================================
# WAIT FOR SERVER
# ============================================================


def wait_for_server(port):

    start = time.time()

    while time.time() - start < 20:

        try:

            s = socket.create_connection(
                ("127.0.0.1", port),
                timeout=1
            )

            s.close()

            return True

        except:

            time.sleep(0.5)

    return False


print(
    "SERVER READY:",
    wait_for_server(PORT)
)


# ============================================================
# CLOUDFLARE TUNNEL
# ============================================================

print("Starting Cloudflare tunnel...")

cloudflared = subprocess.Popen(
    [
        "cloudflared",
        "tunnel",
        "--url",
        f"http://127.0.0.1:{PORT}"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None

for _ in range(120):

    line = cloudflared.stdout.readline()

    print(line.strip())

    match = re.search(
        r"https://[-a-zA-Z0-9]+\.trycloudflare\.com",
        line
    )

    if match:

        public_url = match.group(0)

        break

    time.sleep(0.2)


# ============================================================
# FINAL OUTPUT
# ============================================================

print()
print("====================================================")
print("LSML LATENT PROJECTOR RUNTIME")
print("====================================================")
print()
print("LOCAL:")
print(f"http://127.0.0.1:{PORT}")
print()
print("PUBLIC:")
print(public_url)
print()
print("====================================================")
print()

while True:

    time.sleep(1000)

DEVICE: cuda


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


PORT: 45099
SERVER READY: True
Starting Cloudflare tunnel...
2026-05-28T08:27:32Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-05-28T08:27:32Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-05-28T08:27:37Z INF +--------------------------------------------------------------------------------------------+
2026-05-28T08:27:37Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-05-28T08:27:37Z

KeyboardInterrupt: 